# Aula 2, FastText

Notebook executável que acompanha a aula [02-fasttext.md](../../lessons/modulo-04-word-embeddings/02-fasttext.md).

## O que vamos fazer aqui

O FastText olha para dentro das palavras, representando-as pelos seus n-gramas de
caracteres. Vamos usar essa ideia para medir similaridade entre palavras e, o mais
importante, mostrar a superpotência do método: dar um vetor razoável a uma palavra
fora do vocabulário, como uma flexão inédita ou um erro de digitação. Python puro.

## Representando palavras por n-gramas de caracteres

Adicionamos marcadores de início e fim a cada palavra e a quebramos em trigramas de
caracteres. A palavra vira a contagem dos seus n-gramas, e a similaridade entre duas
palavras é o cosseno entre essas contagens.

In [ ]:
from collections import Counter
import math


def ngramas(palavra, n=3):
    """N-gramas de caracteres da palavra, com marcadores de início e fim."""
    p = "<" + palavra + ">"
    return [p[i:i + n] for i in range(len(p) - n + 1)]


def vetor_ngram(palavra, n=3):
    return Counter(ngramas(palavra, n))


def cosseno(a, b):
    chaves = set(a) | set(b)
    produto = sum(a.get(k, 0) * b.get(k, 0) for k in chaves)
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return produto / (na * nb) if na and nb else 0.0


print("trigramas de 'corredor':", ngramas("corredor"))

Veja como corredor se decompõe em pedaços como `<co`, `cor`, `orr`, `rre` e
assim por diante. Palavras da mesma família vão compartilhar muitos desses pedaços, e
é isso que vai aproximá-las.

## Similaridade entre palavras, inclusive uma fora do vocabulário

Agora comparamos pares de palavras. Repare especialmente na última linha, com a
palavra corredores, que não tratamos antes: ela deve ficar bem próxima de corredor,
porque divide quase todos os n-gramas.

In [ ]:
def sim(p1, p2):
    return round(cosseno(vetor_ngram(p1), vetor_ngram(p2)), 3)


print("correr ~ corredor :", sim("correr", "corredor"))
print("correr ~ correndo :", sim("correr", "correndo"))
print("gato ~ gatinho    :", sim("gato", "gatinho"))
print("correr ~ gato     :", sim("correr", "gato"))
print("FORA DO VOCAB corredores ~ corredor:", sim("corredores", "corredor"))

Palavras da mesma família, como correr e corredor, ficam com similaridade
alta (perto de 0,58), enquanto palavras sem relação, como correr e gato, dão zero. E o
ponto alto: corredores, que nem estava na nossa lista, fica a cerca de 0,78 de
corredor. É essa capacidade de representar o que nunca foi visto, compondo a partir dos
pedaços, que o Word2Vec não tinha. Para o projeto, construa um corretor que sugere a
forma base de uma palavra errada pela similaridade de n-gramas.